In [42]:
import os
from dotenv import load_dotenv

load_dotenv(".env")

MYSQL = {
    "host":     os.getenv("MYSQL_HOST", "localhost"),
    "port":     int(os.getenv("MYSQL_PORT", 3306)),
    "user":     os.getenv("MYSQL_USER", "root"),
    "password": os.getenv("MYSQL_PASSWORD"),
    "database": os.getenv("MYSQL_DATABASE", "logistics_db"),
}

MONGO = {
    "host":       os.getenv("MONGO_HOST", "localhost"),
    "port":       int(os.getenv("MONGO_PORT", 27017)),
    "database":   os.getenv("MONGO_DATABASE", "logistics_db"),
    "collection": os.getenv("MONGO_COLLECTION", "delivery_events"),
}

MYSQL_URL = (
    f"jdbc:mysql://{MYSQL['host']}:{MYSQL['port']}/{MYSQL['database']}"
    f"?useSSL=false&allowPublicKeyRetrieval=true"
)
MONGO_URI = (
    f"mongodb://{MONGO['host']}:{MONGO['port']}/"
    f"{MONGO['database']}.{MONGO['collection']}"
)
PACKAGES = (
    "mysql:mysql-connector-java:8.0.33,"
    "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0"
)

DATA_PROCESSED = "file://" + os.path.abspath(os.path.join("..", "data", "processed"))

print("Config loaded from .env")
print(f"MySQL : {MYSQL['user']}@{MYSQL['host']}:{MYSQL['port']}/{MYSQL['database']}")
print(f"Mongo : {MONGO['host']}:{MONGO['port']}/{MONGO['database']}")

Config loaded from .env
MySQL : root@localhost:3306/logistics_db
Mongo : localhost:27017/logistics_db


In [43]:
# Start Spark Session

from pyspark.sql import SparkSession

# Stop any existing session first
try:
    spark.stop()
except:
    pass

spark = (
    SparkSession.builder
    .appName("LogisticsETL")
    .master("local[*]")
    .config("spark.jars.packages",              PACKAGES)
    .config("spark.mongodb.read.connection.uri", MONGO_URI)
    .config("spark.sql.shuffle.partitions",      "8")
    .config("spark.driver.memory",               "2g")
    .config("spark.driver.host",                 "127.0.0.1")
    .config("spark.driver.bindAddress",          "127.0.0.1")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} ready")

26/04/28 09:27:57 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark 4.1.1 ready


In [44]:
# Extract from MySQL

def read_mysql(table):
    return (
        spark.read.format("jdbc")
        .option("url",      MYSQL_URL)
        .option("dbtable",  table)
        .option("user",     MYSQL["user"])
        .option("password", MYSQL["password"])
        .option("driver",   "com.mysql.cj.jdbc.Driver")
        .load()
    )

df_orders  = read_mysql("orders")
df_drivers = read_mysql("drivers")

print(f"orders  : {df_orders.count():,} rows")
print(f"drivers : {df_drivers.count():,} rows")
df_orders.printSchema()

orders  : 45,593 rows
drivers : 1,320 rows
root
 |-- order_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- restaurant_lat: decimal(10,6) (nullable = true)
 |-- restaurant_lng: decimal(10,6) (nullable = true)
 |-- delivery_lat: decimal(10,6) (nullable = true)
 |-- delivery_lng: decimal(10,6) (nullable = true)
 |-- order_date: date (nullable = true)
 |-- time_ordered: timestamp (nullable = true)
 |-- time_order_picked: timestamp (nullable = true)
 |-- weather_conditions: string (nullable = true)
 |-- road_traffic_density: string (nullable = true)
 |-- vehicle_condition: integer (nullable = true)
 |-- type_of_order: string (nullable = true)
 |-- type_of_vehicle: string (nullable = true)
 |-- multiple_deliveries: integer (nullable = true)
 |-- festival: string (nullable = true)
 |-- city: string (nullable = true)
 |-- time_taken_min: integer (nullable = true)



In [45]:
# Extract from MongoDB

import pymongo
import pandas as pd

# Load from MongoDB via pymongo
client = pymongo.MongoClient(MONGO["host"], MONGO["port"])
col    = client[MONGO["database"]][MONGO["collection"]]

docs = list(col.find({}, {"_id": 0}))
client.close()

# Convert to Spark DataFrame
df_events = spark.createDataFrame(pd.DataFrame(docs))

print(f"delivery_events : {df_events.count():,} documents")
df_events.printSchema()

delivery_events : 42,247 documents
root
 |-- order_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- location: map (nullable = true)
 |    |-- key: string
 |    |-- value: double (valueContainsNull = true)
 |-- notes: string (nullable = true)
 |-- metadata: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)



In [46]:
# Transform Orders

import math
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

@F.udf(DoubleType())
def haversine(lat1, lon1, lat2, lon2):
    if None in (lat1, lon1, lat2, lon2): return None
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a  = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return R * 2 * math.asin(math.sqrt(a))

df_ord = (
    df_orders
    .withColumn("time_taken_min",      F.col("time_taken_min").cast(DoubleType()))
    .withColumn("vehicle_condition",   F.col("vehicle_condition").cast(IntegerType()))
    .withColumn("multiple_deliveries", F.col("multiple_deliveries").cast(IntegerType()))
    .withColumn("restaurant_lat",      F.col("restaurant_lat").cast(DoubleType()))
    .withColumn("restaurant_lng",      F.col("restaurant_lng").cast(DoubleType()))
    .withColumn("delivery_lat",        F.col("delivery_lat").cast(DoubleType()))
    .withColumn("delivery_lng",        F.col("delivery_lng").cast(DoubleType()))
    .filter(F.col("time_taken_min").isNotNull())
)

# Normalise categoricals to lowercase
for c in ["weather_conditions", "road_traffic_density",
          "type_of_order", "type_of_vehicle", "city"]:
    df_ord = df_ord.withColumn(c, F.lower(F.trim(F.col(c))))

# Join driver info
df_ord = df_ord.join(
    df_drivers.select("driver_id", "age", "rating"),
    on="driver_id", how="left"
)

# Compute haversine distance
df_ord = df_ord.withColumn(
    "distance_km",
    haversine(
        F.col("restaurant_lat"), F.col("restaurant_lng"),
        F.col("delivery_lat"),   F.col("delivery_lng")
    )
)

# Delivery speed
df_ord = df_ord.withColumn(
    "speed_kmpm",
    F.when(F.col("time_taken_min") > 0,
           F.col("distance_km") / F.col("time_taken_min"))
    .otherwise(None)
)

# Delay bucket
df_ord = df_ord.withColumn(
    "delay_bucket",
    F.when(F.col("time_taken_min") <  20, "fast")
    .when( F.col("time_taken_min") <  40, "normal")
    .when( F.col("time_taken_min") <  60, "slow")
    .otherwise("very_slow")
)

# Hour of day
df_ord = df_ord.withColumn(
    "hour_of_day",
    F.hour(F.to_timestamp(F.col("time_ordered"), "HH:mm:ss"))
)

print(f"Transformed orders: {df_ord.count():,} rows")
df_ord.show(3, truncate=False)

Transformed orders: 45,593 rows
+---------------+--------+--------------+--------------+------------+------------+----------+-------------------+-------------------+------------------+--------------------+-----------------+-------------+---------------+-------------------+--------+-------------+--------------+---+------+------------------+-------------------+------------+-----------+
|driver_id      |order_id|restaurant_lat|restaurant_lng|delivery_lat|delivery_lng|order_date|time_ordered       |time_order_picked  |weather_conditions|road_traffic_density|vehicle_condition|type_of_order|type_of_vehicle|multiple_deliveries|festival|city         |time_taken_min|age|rating|distance_km       |speed_kmpm         |delay_bucket|hour_of_day|
+---------------+--------+--------------+--------------+------------+------------+----------+-------------------+-------------------+------------------+--------------------+-----------------+-------------+---------------+-------------------+--------+--------

In [47]:
# Transform Events

df_ev = (
    df_events
    .select(
        F.col("order_id"),
        F.col("driver_id"),
        F.col("event_type"),
        F.col("timestamp").cast("timestamp").alias("event_ts"),
        F.col("location.lat").cast(DoubleType()).alias("event_lat"),
        F.col("location.lng").cast(DoubleType()).alias("event_lng"),
        F.col("notes"),
    )
    .filter(F.col("order_id").isNotNull())
    .withColumn("event_date", F.to_date("event_ts"))
    .withColumn("event_hour", F.hour("event_ts"))
    .withColumn("stage",
        F.when(F.col("event_type") == "order_placed", 1)
        .when( F.col("event_type") == "picked_up",    2)
        .when( F.col("event_type") == "in_transit",   3)
        .when( F.col("event_type") == "delivered",    4)
        .otherwise(0))
)

print(f"Transformed events: {df_ev.count():,} rows")
df_ev.show(3, truncate=False)

Transformed events: 42,247 rows
+--------+---------------+------------+-------------------+---------+---------+-----+----------+----------+-----+
|order_id|driver_id      |event_type  |event_ts           |event_lat|event_lng|notes|event_date|event_hour|stage|
+--------+---------------+------------+-------------------+---------+---------+-----+----------+----------+-----+
|0x2318  |COIMBRES13DEL01|order_placed|2022-03-30 08:00:00|11.003669|76.976494|NULL |2022-03-30|8         |1    |
|0x2318  |COIMBRES13DEL01|picked_up   |2022-03-30 15:05:00|11.002319|76.974833|NULL |2022-03-30|15        |2    |
|0x2318  |COIMBRES13DEL01|delivered   |2022-03-30 09:00:00|11.044632|77.016675|NULL |2022-03-30|9         |4    |
+--------+---------------+------------+-------------------+---------+---------+-----+----------+----------+-----+
only showing top 3 rows


Traceback (most recent call last):
  File "/Users/heangpiseth/miniconda3/envs/itc/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/heangpiseth/miniconda3/envs/itc/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


In [48]:
import os

DATA_PROCESSED = "file://" + os.path.abspath(os.path.join("..", "data", "processed"))

path_orders = DATA_PROCESSED + "/orders_clean"
path_events = DATA_PROCESSED + "/events_clean"

df_ord.write.mode("overwrite").parquet(path_orders)
df_ev.write.mode("overwrite").parquet(path_events)

print(f"Saved orders: {path_orders}")
print(f"Saved events: {path_events}")

Saved orders: file:///Users/heangpiseth/Desktop/ITC/I4 AMS/Data Engineering/data/processed/orders_clean
Saved events: file:///Users/heangpiseth/Desktop/ITC/I4 AMS/Data Engineering/data/processed/events_clean


In [49]:
# Verify

check_orders = spark.read.parquet(path_orders)
check_events = spark.read.parquet(path_events)

print(f"orders_clean : {check_orders.count():,} rows")
print(f"events_clean : {check_events.count():,} rows")

print("\nSample orders:")
check_orders.show(3, truncate=False)

spark.stop()
print("\nNotebook 4 complete!")

orders_clean : 45,593 rows
events_clean : 42,247 rows

Sample orders:
+----------------+--------+--------------+--------------+------------+------------+----------+-------------------+-------------------+------------------+--------------------+-----------------+-------------+----------------+-------------------+--------+-------------+--------------+---+------+------------------+-------------------+------------+-----------+
|driver_id       |order_id|restaurant_lat|restaurant_lng|delivery_lat|delivery_lng|order_date|time_ordered       |time_order_picked  |weather_conditions|road_traffic_density|vehicle_condition|type_of_order|type_of_vehicle |multiple_deliveries|festival|city         |time_taken_min|age|rating|distance_km       |speed_kmpm         |delay_bucket|hour_of_day|
+----------------+--------+--------------+--------------+------------+------------+----------+-------------------+-------------------+------------------+--------------------+-----------------+-------------+----------